# Phase 05 · Bài 07 — POS Tagging và Syntactic Parsing

**Mục tiêu:** hiểu POS tagging như bài gán nhãn chuỗi; tự huấn luyện baseline most-frequent-tag và bigram HMM, giải mã bằng Viterbi; đọc dependency tree và trích triple; thấy giới hạn của mô hình cục bộ.

## 1. Vì sao bài này quay lại

Bài 01 đã hứa rằng lemmatization cần POS: không biết `running` là động từ thì không thể đưa về `run`; không biết `better` là tính từ thì không thể đưa về `good`. Lời hứa đó che cả một phân ngành.

POS tagging gán phạm trù ngữ pháp cho từng token. Syntactic parsing khôi phục cấu trúc cây của câu: từ nào bổ nghĩa cho từ nào, động từ nào chi phối tham tố nào. NLP cổ điển dành hai mươi năm tinh chỉnh cả hai; deep learning gộp chúng thành bài token classification trên transformer, và cộng đồng nghiên cứu chuyển hướng.

Cộng đồng ứng dụng thì không. Mọi pipeline trích xuất có cấu trúc vẫn dùng POS và dependency tree: validate JSON do LLM sinh ra theo ràng buộc ngữ pháp, phân rã câu hỏi, kiểm tra alignment của parse tree trong đánh giá dịch máy, gán cảm xúc theo khía cạnh.

## 2. Tagset

```text
The/DET cats/NOUN were/AUX running/VERB at/ADP 3pm/NOUN ./PUNCT
```

- **Penn Treebank (PTB)**: 36 tag cho tiếng Anh, phân biệt `NN` (danh từ số ít), `NNS` (số nhiều), `NNP` (danh từ riêng), `VBD` (quá khứ), `VBZ` (ngôi 3 số ít hiện tại)…
- **Universal Dependencies (UD)**: 17 tag, ngôn ngữ trung lập, mặc định cho cross-lingual.

Chọn tagset là quyết định hệ thống: PTB giữ nhiều thông tin hình thái tiếng Anh nhưng không chuyển sang ngôn ngữ khác; UD thô hơn nhưng dùng được cho hàng trăm ngôn ngữ.

## 3. Baseline most-frequent-tag

Với mỗi từ, dự đoán tag xuất hiện nhiều nhất trong training. Trên Brown corpus, baseline này đạt khoảng 85% accuracy. Đó là sàn: không mô hình nghiêm túc nào nên thấp hơn.

Baseline này thất bại ở đúng chỗ dễ đoán: từ mơ hồ. `book` trong `the book` là NOUN, trong `they book flights` là VERB. MFT chọn một tag cho mọi ngữ cảnh.

## 4. Bigram HMM

Mô hình xác suất đồng thời của chuỗi:

P(tags, words) = Π P(tag_i | tag_{i-1}) · P(word_i | tag_i)

Hai bảng: transition (tag sau | tag trước) và emission (word | tag), ước lượng bằng đếm cộng làm trơn Laplace. Giải mã bằng Viterbi — quy hoạch động trên lưới tag.

Viterbi giữ, với mỗi vị trí i và mỗi tag j, điểm tốt nhất của đường đi kết thúc tại j:

V[i][j] = emission(i, j) + max_k ( V[i-1][k] + transition(k, j) )

Con trỏ `back[i][j]` ghi lại k tốt nhất để truy ngược đường đi. Chi phí O(n·|T|²), với |T| khoảng 17–36 tag là rất nhỏ.

Bigram HMM trên Brown đạt khoảng 93%. Bước nhảy 85 → 93 chủ yếu nhờ transition: mô hình học `DET NOUN` phổ biến và `NOUN DET` hiếm. Đây là lý do một từ OOV sau `DET` vẫn được gán NOUN dù emission không có thông tin.

## 5. Vì sao tagger hiện đại vượt qua

Transition và emission đều cục bộ. Chúng không biết `saw` là NOUN trong `I bought a saw` nhưng là VERB trong `I saw the movie` nếu hai ngữ cảnh có cùng tập từ lân cận. CRF với đặc trưng tùy ý (hậu tố, word shape, từ trước/sau, chính từ) đạt khoảng 97%. BiLSTM-CRF hoặc transformer đạt 98%+.

Trần của bài toán do bất đồng giữa người gán nhãn quyết định: trên Penn Treebank, người gán nhãn đồng ý với nhau khoảng 97% trường hợp. Mô hình trên 98% có thể đang overfit test set, không phải thực sự tốt hơn con người.

## 6. Parsing

Hai họ cổ điển:

- **Transition-based** (arc-eager, arc-standard): đọc token, đẩy vào stack, áp dụng action shift/reduce tạo cung. Giải mã tham lam, nhanh. MaltParser là bản cổ điển; Chen & Manning là bản neural.
- **Graph-based** (Eisner, Dozat-Manning biaffine): chấm điểm mọi cặp head–dependent, chọn maximum spanning tree. Chậm hơn, chính xác hơn.

Dependency parsing thắng trong thập niên 2010 vì tổng quát hóa sạch sang ngôn ngữ trật tự tự do. Mỗi từ có đúng một head; mỗi cạnh là bộ ba (head, dependent, relation).

```text
running là ROOT
cats là nsubj của running
were là aux của running
at là prep của running
3pm là pobj của at
```

Đọc cột `dep` từ dưới lên là cấu trúc ngữ pháp của câu hiện ra.

## 7. Đo lường và bẫy

- **POS**: accuracy theo token, nhưng nên báo per-tag F1 vì tag hiếm (`,` `$` `#`) bị pha loãng.
- **Dependency**: UAS (unlabeled attachment score) và LAS (labeled). LAS khó hơn vì phải đúng cả quan hệ.
- **Bẫy kinh điển**: mơ hồ noun–verb (`saw`, `book`, `can`) và PP-attachment (`I saw the man with the telescope`). Luôn lấy mẫu 20 output và đọc bằng mắt.

## 8. Còn dùng ở đâu (2026)

- Lemmatization cần POS. Luôn luôn.
- Validate đầu ra LLM theo ràng buộc ngữ pháp.
- Aspect-based sentiment: parse cho biết tính từ bổ nghĩa cho danh từ nào.
- Query understanding: `movies directed by Wes Anderson starring Bill Murray` phân rã thành ràng buộc có cấu trúc.
- Cross-lingual transfer nhờ UD.
- Pipeline ít compute: POS + dependency + gazetteer đi được khá xa.

Không nên tự viết parser từ đầu cho ứng dụng. Đó là dự án nghiên cứu, không phải task ứng dụng.


## A. Chạy code mẫu của bài

Cell dưới chạy `code/main.py` của bài 07: huấn luyện most-frequent-tag và bigram HMM trên 6 câu, sau đó gán nhãn 3 câu test bằng cả hai để so trực tiếp.


In [1]:
from pathlib import Path
import subprocess
import sys

PHASE = "phases/05-nlp-foundations-to-advanced"
LESSON = "07-pos-tagging-parsing"
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / PHASE / "README.md").is_file():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Mở Jupyter từ thư mục repo hoặc thư mục con của repo.")
script = REPO / PHASE / LESSON / "code" / "main.py"
result = subprocess.run([sys.executable, str(script)], cwd=REPO, text=True, capture_output=True)
print("Bài:", "POS Tagging and Syntactic Parsing")
print("Lệnh:", script.relative_to(REPO))
print("Python:", sys.version.split()[0])
print("Exit code:", result.returncode)
print("\n--- stdout ---")
print(result.stdout.rstrip())
if result.stderr:
    print("\n--- stderr ---")
    print(result.stderr.rstrip())
if result.returncode != 0:
    raise RuntimeError(f"Code mẫu của {LESSON} thoát với mã {result.returncode}")


Bài: POS Tagging and Syntactic Parsing
Lệnh: phases\05-nlp-foundations-to-advanced\07-pos-tagging-parsing\code\main.py
Python: 3.12.10
Exit code: 0

--- stdout ---
tokens: ['The', 'cat', 'chased', 'the', 'dog']
  mft : ['DET', 'NOUN', 'NOUN', 'DET', 'NOUN']
  hmm : ['DET', 'NOUN', 'AUX', 'DET', 'NOUN']

tokens: ['A', 'red', 'mat', 'is', 'here']
  mft : ['DET', 'ADJ', 'NOUN', 'AUX', 'NOUN']
  hmm : ['DET', 'ADJ', 'NOUN', 'AUX', 'ADJ']

tokens: ['Dogs', 'bark']
  mft : ['NOUN', 'VERB']
  hmm : ['NOUN', 'VERB']


## B. Thực hành riêng — đo accuracy và tìm đúng chỗ HMM thắng

Code mẫu chỉ in nhãn, không đo. Ta huấn luyện trên 12 câu có tag, đánh giá trên 6 câu giữ riêng, và tính:

- token accuracy của MFT và HMM,
- per-tag precision/recall/F1 của HMM,
- các cặp nhầm nhiều nhất,
- transition xác suất cao nhất từ `DET` và từ `NOUN`.

Trong test có hai từ chưa từng xuất hiện (`zorbat`) đứng ở vị trí động từ. MFT rơi về tag mặc định (thường là NOUN). HMM dùng transition `NOUN → VERB` và gán đúng. Đây chính là cơ chế đã đưa HMM từ 85% lên 93% trên Brown: ngữ cảnh tag lân cận bù cho emission không có thông tin.

Mã nguồn sắp xếp danh sách tag trước khi giải mã để kết quả tất định giữa các lần chạy (thứ tự `set` của Python phụ thuộc hash seed).


In [2]:
import math
from collections import Counter, defaultdict

DET, NOUN, VERB, ADJ, ADV, ADP, AUX, PRON, PUNCT = "DET", "NOUN", "VERB", "ADJ", "ADV", "ADP", "AUX", "PRON", "PUNCT"

TRAIN = [
    ("The cat sat on the mat .".split(), [DET, NOUN, VERB, ADP, DET, NOUN, PUNCT]),
    ("A dog ran across the road .".split(), [DET, NOUN, VERB, ADP, DET, NOUN, PUNCT]),
    ("Cats chase mice .".split(), [NOUN, VERB, NOUN, PUNCT]),
    ("Dogs bark loudly .".split(), [NOUN, VERB, ADV, PUNCT]),
    ("The mat is red .".split(), [DET, NOUN, AUX, ADJ, PUNCT]),
    ("A red cat sat .".split(), [DET, ADJ, NOUN, VERB, PUNCT]),
    ("She reads books quickly .".split(), [PRON, VERB, NOUN, ADV, PUNCT]),
    ("He writes code every day .".split(), [PRON, VERB, NOUN, DET, NOUN, PUNCT]),
    ("The old man walks slowly .".split(), [DET, ADJ, NOUN, VERB, ADV, PUNCT]),
    ("They play music loudly .".split(), [PRON, VERB, NOUN, ADV, PUNCT]),
    ("We bought a new car .".split(), [PRON, VERB, DET, ADJ, NOUN, PUNCT]),
    ("The children eat apples .".split(), [DET, NOUN, VERB, NOUN, PUNCT]),
]

TEST = [
    ("The cat sat on the mat .".split(), [DET, NOUN, VERB, ADP, DET, NOUN, PUNCT]),
    ("Cats zorbat mice .".split(), [NOUN, VERB, NOUN, PUNCT]),
    ("The old man walks slowly .".split(), [DET, ADJ, NOUN, VERB, ADV, PUNCT]),
    ("We bought a new car .".split(), [PRON, VERB, DET, ADJ, NOUN, PUNCT]),
    ("She reads books quickly .".split(), [PRON, VERB, NOUN, ADV, PUNCT]),
    ("Dogs zorbat loudly .".split(), [NOUN, VERB, ADV, PUNCT]),
]


def train_mft(examples):
    word_tag_counts = defaultdict(Counter)
    all_tags = Counter()
    for tokens, tags in examples:
        for token, tag in zip(tokens, tags):
            word_tag_counts[token.lower()][tag] += 1
            all_tags[tag] += 1
    word_best = {word: counts.most_common(1)[0][0] for word, counts in word_tag_counts.items()}
    default_tag = all_tags.most_common(1)[0][0]
    return word_best, default_tag


def predict_mft(tokens, word_best, default_tag):
    return [word_best.get(token.lower(), default_tag) for token in tokens]


def train_hmm(examples, alpha=0.01):
    transitions = defaultdict(Counter)
    emissions = defaultdict(Counter)
    tags, vocab = set(), set()
    for tokens, gold in examples:
        previous = "<BOS>"
        for token, tag in zip(tokens, gold):
            transitions[previous][tag] += 1
            emissions[tag][token.lower()] += 1
            tags.add(tag)
            vocab.add(token.lower())
            previous = tag
        transitions[previous]["<EOS>"] += 1
    return transitions, emissions, sorted(tags), vocab


def log_prob(counts, key, denominator, alpha=0.01):
    return math.log((counts.get(key, 0) + alpha) / denominator)


def viterbi(tokens, transitions, emissions, tags, vocab, alpha=0.01):
    n = len(tokens)
    emission_denominator = {tag: sum(emissions[tag].values()) + alpha * (len(vocab) + 1) for tag in tags}
    transition_denominator = {previous: sum(transitions[previous].values()) + alpha * (len(tags) + 1)
                              for previous in list(transitions) + ["<BOS>"]}
    scores = []
    back = []
    first = {}
    for tag in tags:
        first[tag] = log_prob(transitions["<BOS>"], tag, transition_denominator["<BOS>"], alpha) + \
            log_prob(emissions[tag], tokens[0].lower(), emission_denominator[tag], alpha)
    scores.append(first)
    back.append({tag: None for tag in tags})
    for i in range(1, n):
        row, pointers = {}, {}
        for tag in tags:
            emission = log_prob(emissions[tag], tokens[i].lower(), emission_denominator[tag], alpha)
            best_score, best_previous = -math.inf, None
            for previous in tags:
                score = scores[i - 1][previous] + \
                    log_prob(transitions[previous], tag, transition_denominator[previous], alpha) + emission
                if score > best_score:
                    best_score, best_previous = score, previous
            row[tag] = best_score
            pointers[tag] = best_previous
        scores.append(row)
        back.append(pointers)
    last = max(tags, key=lambda tag: scores[-1][tag])
    path = [last]
    for i in range(n - 1, 0, -1):
        path.append(back[i][path[-1]])
    return list(reversed(path))


word_best, default_tag = train_mft(TRAIN)
transitions, emissions, tags, vocab = train_hmm(TRAIN)
print("tag mặc định (phổ biến nhất):", default_tag)
print("số tag:", len(tags), "| kích thước vocab:", len(vocab))

mft_correct = hmm_correct = total = 0
confusions = Counter()
per_tag = {tag: {"tp": 0, "fp": 0, "fn": 0} for tag in tags}
print(f"\n{'câu':44s} {'MFT':>6s} {'HMM':>6s}")
for tokens, gold in TEST:
    mft = predict_mft(tokens, word_best, default_tag)
    hmm = viterbi(tokens, transitions, emissions, tags, vocab)
    mft_hits = sum(1 for g, p in zip(gold, mft) if g == p)
    hmm_hits = sum(1 for g, p in zip(gold, hmm) if g == p)
    mft_correct += mft_hits
    hmm_correct += hmm_hits
    total += len(gold)
    for g, p in zip(gold, hmm):
        if g == p:
            per_tag[g]["tp"] += 1
        else:
            per_tag[g]["fn"] += 1
            per_tag[p]["fp"] += 1
            confusions[(g, p)] += 1
    print(f"{' '.join(tokens):44s} {mft_hits:>3d}/{len(gold)} {hmm_hits:>3d}/{len(gold)}")

print(f"\nMFT accuracy = {mft_correct / total:.3f}")
print(f"HMM accuracy = {hmm_correct / total:.3f}")

print("\nHMM per-tag:")
print(f"{'tag':6s} {'P':>6s} {'R':>6s} {'F1':>6s}")
for tag in tags:
    tp, fp, fn = per_tag[tag]["tp"], per_tag[tag]["fp"], per_tag[tag]["fn"]
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    if tp + fp + fn:
        print(f"{tag:6s} {precision:6.3f} {recall:6.3f} {f1:6.3f}")

print("\ncác cặp nhầm của HMM:")
for (gold_tag, predicted_tag), count in confusions.most_common():
    print(f"  gold={gold_tag:5s} pred={predicted_tag:5s} x{count}")

for tag in ("<BOS>", DET, NOUN):
    ranked = transitions[tag].most_common(3)
    total_tag = sum(transitions[tag].values())
    formatted = ", ".join(f"{nxt}:{count / total_tag:.2f}" for nxt, count in ranked)
    print(f"\ntransition từ {tag:5s} -> {formatted}")

print("\nTừ OOV 'zorbat' đứng sau NOUN: MFT dùng tag mặc định, HMM dùng transition NOUN -> VERB.")


tag mặc định (phổ biến nhất): NOUN
số tag: 9 | kích thước vocab: 42

câu                                             MFT    HMM
The cat sat on the mat .                       7/7   7/7
Cats zorbat mice .                             3/4   4/4
The old man walks slowly .                     6/6   6/6
We bought a new car .                          6/6   6/6
She reads books quickly .                      5/5   5/5
Dogs zorbat loudly .                           3/4   4/4

MFT accuracy = 0.938
HMM accuracy = 1.000

HMM per-tag:
tag         P      R     F1
ADJ     1.000  1.000  1.000
ADP     1.000  1.000  1.000
ADV     1.000  1.000  1.000
DET     1.000  1.000  1.000
NOUN    1.000  1.000  1.000
PRON    1.000  1.000  1.000
PUNCT   1.000  1.000  1.000
VERB    1.000  1.000  1.000

các cặp nhầm của HMM:

transition từ <BOS> -> DET:0.50, PRON:0.33, NOUN:0.17

transition từ DET   -> NOUN:0.70, ADJ:0.30

transition từ NOUN  -> VERB:0.41, PUNCT:0.35, ADV:0.12

Từ OOV 'zorbat' đứng sau NOUN: MFT dùng ta

## C. Thí nghiệm — đọc dependency tree và trích triple

Phần này không huấn luyện parser. Ta dựng một dependency tree bằng tay (đúng định dạng đầu ra của spaCy: mỗi token có head và relation), rồi viết đoạn code ứng dụng thật sự cần: trích bộ ba subject–verb–object, và nhận diện chỗ parse sai.

Ba điểm cần thấy:

1. **Cấu trúc nằm ở cạnh, không ở token.** `nsubj`, `dobj`, `prep` cho biết ai làm gì với ai.
2. **Passive phá extractor ngây thơ.** Trong `The startup was bought by the company`, chủ thể ngữ nghĩa nằm ở `by`, không phải `nsubjpass`. Extractor đơn giản sẽ trả object rỗng.
3. **PP-attachment mơ hồ.** `I saw the man with the telescope`: `with` bổ nghĩa cho `saw` (dùng kính) hay cho `man` (người có kính)? Hai parse đều hợp lệ ngữ pháp; chỉ ngữ nghĩa mới phân xử.

Cuối cùng là ví dụ lemma phụ thuộc POS: cùng chuỗi `running`, POS khác nhau cho lemma khác nhau.


In [3]:
def build_tree(tokens, arcs, pos, root):
    head = {i: i for i in range(len(tokens))}
    relation = {i: "root" for i in range(len(tokens))}
    children = {i: [] for i in range(len(tokens))}
    for head_index, dep_index, rel in arcs:
        head[dep_index] = head_index
        relation[dep_index] = rel
        children[head_index].append((dep_index, rel))
    return {"tokens": tokens, "pos": pos, "head": head, "relation": relation, "children": children, "root": root}


def show_tree(tree):
    print(f"{'i':>2s} {'token':12s} {'pos':6s} {'dep':10s} {'head':12s}")
    for i, token in enumerate(tree["tokens"]):
        head_index = tree["head"][i]
        marker = "ROOT" if i == tree["root"] else tree["tokens"][head_index]
        print(f"{i:>2d} {token:12s} {tree['pos'][i]:6s} {tree['relation'][i]:10s} {marker:12s}")


def extract_svo(tree):
    verbs = [i for i, tag in enumerate(tree["pos"]) if tag == "VERB"]
    triples = []
    for verb in verbs:
        subject = object_ = None
        for child, rel in tree["children"][verb]:
            if rel == "nsubj":
                subject = " ".join(tree["tokens"][child:child + 1])
            elif rel == "dobj":
                object_ = " ".join(tree["tokens"][child:child + 1])
        triples.append((subject, tree["tokens"][verb], object_))
    return triples


tokens = "The company bought a small startup in Berlin .".split()
pos = ["DET", "NOUN", "VERB", "DET", "ADJ", "NOUN", "ADP", "NOUN", "PUNCT"]
arcs = [
    (1, 0, "det"),
    (2, 1, "nsubj"),
    (2, 2, "root"),
    (5, 3, "det"),
    (5, 4, "amod"),
    (2, 5, "dobj"),
    (2, 6, "prep"),
    (6, 7, "pobj"),
    (2, 8, "punct"),
]
tree = build_tree(tokens, arcs, pos, root=2)
print("=== dependency tree ===")
show_tree(tree)
print("\nSVO:", extract_svo(tree))
print("'in Berlin' là cụm giới từ bổ nghĩa cho 'bought', không phải cho 'startup'.")

print("\n=== passive voice ===")
tokens_passive = "The startup was bought by the company .".split()
pos_passive = ["DET", "NOUN", "AUX", "VERB", "ADP", "DET", "NOUN", "PUNCT"]
arcs_passive = [
    (1, 0, "det"),
    (3, 1, "nsubjpass"),
    (3, 2, "auxpass"),
    (3, 3, "root"),
    (3, 4, "agent"),
    (6, 5, "det"),
    (4, 6, "pobj"),
    (3, 7, "punct"),
]
tree_passive = build_tree(tokens_passive, arcs_passive, pos_passive, root=3)
show_tree(tree_passive)
print("SVO ngây thơ:", extract_svo(tree_passive))
print("Extractor chỉ đọc nsubj/dobj nên trả object rỗng; muốn đúng phải xử lý 'agent' + 'nsubjpass'.")

print("\n=== PP-attachment ambiguity ===")
tokens_pp = "I saw the man with the telescope .".split()
pos_pp = ["PRON", "VERB", "DET", "NOUN", "ADP", "DET", "NOUN", "PUNCT"]
parse_a = [
    (1, 0, "nsubj"),
    (1, 1, "root"),
    (3, 2, "det"),
    (1, 3, "dobj"),
    (1, 4, "prep"),
    (6, 5, "det"),
    (4, 6, "pobj"),
    (1, 7, "punct"),
]
parse_b = [
    (1, 0, "nsubj"),
    (1, 1, "root"),
    (3, 2, "det"),
    (1, 3, "dobj"),
    (3, 4, "prep"),
    (6, 5, "det"),
    (4, 6, "pobj"),
    (1, 7, "punct"),
]
for name, arc_list in (("with -> saw  (dụng cụ)", parse_a), ("with -> man  (sở hữu)", parse_b)):
    tree_pp = build_tree(tokens_pp, arc_list, pos_pp, root=1)
    head_of_with = tree_pp["tokens"][tree_pp["head"][4]]
    print(f"  {name:26s} head của 'with' = {head_of_with!r}")

print("\n=== lemma phụ thuộc POS ===")
LEMMA = {"running": {"VERB": "run", "NOUN": "running"}}
for word, tag in (("running", "VERB"), ("running", "NOUN")):
    print(f"  {word!r} + {tag:4s} -> {LEMMA[word][tag]!r}")
print("Không có POS, không thể chọn lemma đúng.")


=== dependency tree ===
 i token        pos    dep        head        
 0 The          DET    det        company     
 1 company      NOUN   nsubj      bought      
 2 bought       VERB   root       ROOT        
 3 a            DET    det        startup     
 4 small        ADJ    amod       startup     
 5 startup      NOUN   dobj       bought      
 6 in           ADP    prep       bought      
 7 Berlin       NOUN   pobj       in          
 8 .            PUNCT  punct      bought      

SVO: [('company', 'bought', 'startup')]
'in Berlin' là cụm giới từ bổ nghĩa cho 'bought', không phải cho 'startup'.

=== passive voice ===
 i token        pos    dep        head        
 0 The          DET    det        startup     
 1 startup      NOUN   nsubjpass  bought      
 2 was          AUX    auxpass    bought      
 3 bought       VERB   root       ROOT        
 4 by           ADP    agent      bought      
 5 the          DET    det        company     
 6 company      NOUN   pobj       by 

## D. Tự kiểm tra

1. Vì sao MFT là sàn chứ không phải giải pháp? Nó sai ở loại từ nào?
2. Viết công thức Viterbi cho bigram HMM. `back[i][j]` lưu gì?
3. Vì sao HMM gán đúng tag cho một từ OOV đứng sau `DET`?
4. Vì sao trần của POS tagging khoảng 97–98%?
5. Dependency tree khác constituency tree ở điểm nào?
6. Vì sao extractor SVO ngây thơ thất bại trên câu bị động, và cần thêm quan hệ nào?
7. PP-attachment ambiguity ảnh hưởng gì đến một pipeline trích xuất thông tin?

**Giới hạn:** corpus 12 câu là đồ chơi; accuracy ở đây không so sánh được với Brown/UD. Các dependency tree ở phần C được dựng tay để minh họa cách đọc và khai thác đầu ra, không phải kết quả của parser đã huấn luyện. Muốn số liệu thật, chạy spaCy/stanza trên treebank chuẩn.
